# Alternative No-Answer Thresholding

In [1]:
%pip install -q evaluate

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

from datasets import Dataset, load_dataset
from evaluate import load as load_metric
from sklearn.metrics import f1_score, precision_recall_fscore_support
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, Trainer, TrainingArguments
from transformers import set_seed

set_seed(42)

In [3]:
import sys
import torch
import transformers

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.version.cuda)

Python: 3.13.16
NumPy: 2.1.3
scikit-learn: 1.6.1
GPU: Tesla T4
PyTorch: 2.11.0+cu130
Transformers: 5.20.2
CUDA: 13.0


# Load validation data and the saved QA model

In [4]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_validation = dataset["validation"].to_pandas()
langlst = ["ar", "ko", "te"]
df_validation_filtered = df_validation[df_validation["lang"].isin(langlst)]

model_source = "multilingual-qa"
tokenizer = AutoTokenizer.from_pretrained(
    model_source, use_fast=True, local_files_only=True
)
model = AutoModelForQuestionAnswering.from_pretrained(
    model_source, local_files_only=True
)
training_args = TrainingArguments(
    output_dir=model_source,
    fp16=True,
    per_device_eval_batch_size=8,
    seed=42,
    full_determinism=True,
    report_to="none",
)
trainer = Trainer(model=model, args=training_args)

README.md:   0%|          | 0.00/9.88k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B / 6.88MB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 1.38MB            

validation.parquet: downloading bytes:           |  0.00B            

...

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15343 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3011 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

# Compute the core model's spans and no-answer scores

In [5]:
MAX_ANSWER_LENGTH = 30

def prepare_examples(examples):
    tokens = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    example_numbers = tokens.pop("overflow_to_sample_mapping")
    tokens["example_index"] = example_numbers

    for i in range(len(tokens["input_ids"])):
        sequence_ids = tokens.sequence_ids(i)
        tokens["offset_mapping"][i] = [
            offset if sequence_ids[j] == 1 else None
            for j, offset in enumerate(tokens["offset_mapping"][i])
        ]

    return tokens

def predict_answers(examples):
    features = examples.map(
        prepare_examples,
        batched=True,
        batch_size=len(examples),
        remove_columns=examples.column_names,
    )
    model_inputs = features.remove_columns(["offset_mapping", "example_index"])
    start_logits, end_logits = trainer.predict(model_inputs).predictions

    null_scores = np.full(len(examples), np.inf)
    span_scores = np.full(len(examples), -np.inf)
    best_answers = [""] * len(examples)

    for feature_index, feature in enumerate(features):
        example_index = feature["example_index"]
        offsets = feature["offset_mapping"]
        starts = start_logits[feature_index]
        ends = end_logits[feature_index]

        cls_index = feature["input_ids"].index(tokenizer.cls_token_id)
        null_scores[example_index] = min(
            null_scores[example_index],
            float(starts[cls_index] + ends[cls_index]),
        )

        for start, start_offset in enumerate(offsets):
            if start_offset is None or start_offset[1] <= start_offset[0]:
                continue
            for end in range(start, min(start + MAX_ANSWER_LENGTH, len(offsets))):
                end_offset = offsets[end]
                if end_offset is None:
                    break
                if end_offset[1] <= end_offset[0]:
                    continue
                score = float(starts[start] + ends[end])
                if score > span_scores[example_index]:
                    span_scores[example_index] = score
                    context = examples[example_index]["context"]
                    best_answers[example_index] = context[start_offset[0]:end_offset[1]]

    assert np.isfinite(null_scores).all()
    assert np.isfinite(span_scores).all()
    return best_answers, null_scores - span_scores

validation_examples = Dataset.from_pandas(df_validation_filtered, preserve_index=False)
best_answers, scores = predict_answers(validation_examples)

Map:   0%|          | 0/1155 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/linear.py:134: UserWarning: Requested unified CUBLASLT workspace size of 1048576 bytes exceeds CUBLAS workspace size of 131072 bytes. Please increase CUBLAS workspace size via CUBLAS_WORKSPACE_CONFIG or decrease requested CUBLASLT_WORKSPACE_SIZE. Otherwise CUBLASLT workspace size will be limited to the CUBLAS workspace size. (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:228.)
  return F.linear(input, self.weight, self.bias)


# Threshold selection

In [6]:
def select_threshold(scores, gold_answerable):
    thresholds = [scores.min() - 1] + list(np.unique(scores))
    best_threshold = thresholds[0]
    best_macro_f1 = -1

    for threshold in thresholds:
        predicted_answerable = scores <= threshold
        macro_f1 = f1_score(
            gold_answerable,
            predicted_answerable,
            labels=[False, True],
            average="macro",
            zero_division=0,
        )

        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            best_threshold = threshold

    return float(best_threshold)

# Apply language-specific thresholds

In [7]:
gold_answerable = df_validation_filtered["answerable"].to_numpy(dtype=bool)
languages = df_validation_filtered["lang"].to_numpy()

language_thresholds = {}

for language in langlst:
    language_rows = languages == language
    language_thresholds[language] = select_threshold(
        scores[language_rows], gold_answerable[language_rows]
    )

print("Language-specific thresholds:", language_thresholds)

language_predicted_answers = [
    answer if score <= language_thresholds[language] else ""
    for answer, score, language in zip(best_answers, scores, languages)
]

Language-specific thresholds: {'ar': -0.5386962890625, 'ko': 2.32470703125, 'te': -1.507354736328125}


# Validation results

In [8]:
squad_metric = load_metric("squad_v2")

results = []

evaluation = df_validation_filtered.copy()
evaluation["prediction"] = language_predicted_answers
evaluation["pred_answerable"] = evaluation["prediction"] != ""

for language in ["overall", "ar", "ko", "te"]:
    group = evaluation
    if language != "overall":
        group = evaluation[evaluation["lang"] == language]
    formatted_predictions = []
    gold_answers = []

    for example_id, example in group.iterrows():
        prediction = {}
        prediction["id"] = str(example_id)
        prediction["prediction_text"] = example["prediction"]
        prediction["no_answer_probability"] = 0.0  # Threshold already applied.
        formatted_predictions.append(prediction)

        answer = {}
        if example["answerable"]:
            answer["text"] = [example["answer"]]
            answer["answer_start"] = [int(example["answer_start"])]
        else:
            answer["text"] = []
            answer["answer_start"] = []
        gold_answer = {}
        gold_answer["id"] = str(example_id)
        gold_answer["answers"] = answer
        gold_answers.append(gold_answer)

    squad_results = squad_metric.compute(
        references=gold_answers, predictions=formatted_predictions
    )
    precision, recall, class_f1, _ = precision_recall_fscore_support(
        group["answerable"], group["pred_answerable"],
        labels=[False, True], zero_division=0,
    )
    results.append({
        "language": language,
        "N": len(group),
        "SQuAD_v2_EM": squad_results["exact"] / 100,
        "SQuAD_v2_token_F1": squad_results["f1"] / 100,
        "unanswerable_P": precision[0],
        "unanswerable_R": recall[0],
        "unanswerable_F1": class_f1[0],
        "answerable_P": precision[1],
        "answerable_R": recall[1],
        "answerable_F1": class_f1[1],
        "answerability_macro_F1": class_f1.mean(),
    })

results_table = pd.DataFrame(results)
display(results_table.round(4))

,language,N,SQuAD_v2_EM,SQuAD_v2_token_F1,unanswerable_P,unanswerable_R,unanswerable_F1,answerable_P,answerable_R,answerable_F1,answerability_macro_F1
0,overall,1155,0.5827,0.6651,0.7229,0.7317,0.7273,0.9555,0.9536,0.9545,0.8409
1,ar,415,0.6096,0.6973,0.8621,0.9615,0.9091,0.9944,0.9780,0.9861,0.9476
2,ko,356,0.5927,0.6871,0.8125,0.6842,0.7429,0.9824,0.9911,0.9867,0.8648
3,te,384,0.5443,0.6100,0.6196,0.6129,0.6162,0.8767,0.8797,0.8782,0.7472


# Challenge set

In [9]:
records = json.loads(Path("challenge.json").read_text(encoding="utf-8"))
challenge = pd.DataFrame(records)
challenge_examples = Dataset.from_list(records)
challenge_answers, challenge_scores = predict_answers(challenge_examples)

predictions = []
for i, row in enumerate(records):
    threshold = language_thresholds[row["lang"]]
    if challenge_scores[i] <= threshold:
        predictions.append(challenge_answers[i])
    else:
        predictions.append("")
challenge["prediction"] = predictions

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

# Challenge metrics

In [10]:
exact_matches = []
token_f1 = []
for i, row in challenge.iterrows():
    prediction = {}
    prediction["id"] = row["id"]
    prediction["prediction_text"] = row["prediction"]
    prediction["no_answer_probability"] = 0.0  # Threshold already applied.

    answer = {}
    if row["answerable"]:
        answer["text"] = [row["answer"]]
        answer["answer_start"] = [row["answer_start"]]
    else:
        answer["text"] = []
        answer["answer_start"] = []
    gold_answer = {}
    gold_answer["id"] = row["id"]
    gold_answer["answers"] = answer

    result = squad_metric.compute(predictions=[prediction], references=[gold_answer])
    exact_matches.append(result["exact"] == 100)
    token_f1.append(result["f1"] / 100)

challenge["correct"] = exact_matches
challenge["token_F1"] = token_f1
challenge["pred_answerable"] = challenge["prediction"] != ""

challenge_rows = []
for language in ["overall", "ar", "ko", "te"]:
    group = challenge
    if language != "overall":
        group = challenge[challenge["lang"] == language]
    precision, recall, class_f1, _ = precision_recall_fscore_support(
        group["answerable"], group["pred_answerable"],
        labels=[False, True], zero_division=0,
    )
    challenge_rows.append({
        "language": language,
        "N": len(group),
        "SQuAD_v2_EM": group["correct"].mean(),
        "SQuAD_v2_token_F1": group["token_F1"].mean(),
        "unanswerable_P": precision[0],
        "unanswerable_R": recall[0],
        "unanswerable_F1": class_f1[0],
        "answerable_P": precision[1],
        "answerable_R": recall[1],
        "answerable_F1": class_f1[1],
        "answerability_macro_F1": class_f1.mean(),
    })

challenge_results = pd.DataFrame(challenge_rows)
display(challenge_results.round(4))

,language,N,SQuAD_v2_EM,SQuAD_v2_token_F1,unanswerable_P,unanswerable_R,unanswerable_F1,answerable_P,answerable_R,answerable_F1,answerability_macro_F1
0,overall,30,0.4,0.4797,1.0,0.2,0.3333,0.5556,1.0,0.7143,0.5238
1,ar,10,0.5,0.5000,0.0,0.0,0.0000,0.5000,1.0,0.6667,0.3333
2,ko,10,0.4,0.4750,1.0,0.2,0.3333,0.5556,1.0,0.7143,0.5238
3,te,10,0.3,0.4641,1.0,0.4,0.5714,0.6250,1.0,0.7692,0.6703


# Challenge paired results

In [11]:
paired_rows = []
for language in ["overall", "ar", "ko", "te"]:
    group = challenge
    if language != "overall":
        group = challenge[challenge["lang"] == language]
    n_answerable = group["answerable"].sum()
    n_unanswerable = (~group["answerable"]).sum()
    correct = group["pred_answerable"] == group["answerable"]
    correct_answerable = correct[group["answerable"]].sum()
    correct_unanswerable = correct[~group["answerable"]].sum()

    correct_pairs = 0
    exact_pairs = 0
    total_pairs = 0
    for pair_id, pair in group.groupby("pair_id", sort=False):
        if correct.loc[pair.index].all():
            correct_pairs += 1
        if pair["correct"].all():
            exact_pairs += 1
        total_pairs += 1

    paired_rows.append({
        "language": language,
        "n_answerable": n_answerable,
        "n_unanswerable": n_unanswerable,
        "correct_answerable": correct_answerable,
        "correct_unanswerable": correct_unanswerable,
        "answerable_R": correct_answerable / n_answerable if n_answerable else 0.0,
        "unanswerable_R": correct_unanswerable / n_unanswerable if n_unanswerable else 0.0,
        "answerability_correct_pairs": correct_pairs,
        "n_pairs": total_pairs,
        "exact_answerable": group.loc[group["answerable"], "correct"].sum(),
        "exact_match_pairs": exact_pairs,
    })

paired_results = pd.DataFrame(paired_rows)
display(paired_results.round(4))

,language,n_answerable,n_unanswerable,correct_answerable,correct_unanswerable,answerable_R,unanswerable_R,answerability_correct_pairs,n_pairs,exact_answerable,exact_match_pairs
0,overall,15,15,15,3,1.0,0.2,3,15,9,1
1,ar,5,5,5,0,1.0,0.0,0,5,5,0
2,ko,5,5,5,1,1.0,0.2,1,5,3,1
3,te,5,5,5,2,1.0,0.4,2,5,1,0


# Challenge errors

In [12]:
for i, row in challenge.iterrows():
    if row["correct"]:
        continue
    print("\nID:", row["id"], "Language:", row["lang"])
    print("Pair:", row["pair_id"])
    print("Question:", row["question"])
    print("Context:", row["context"])
    print("Gold answerable:", row["answerable"])
    print("Gold answer:", repr(row["answer"]))
    print("Predicted answer:", repr(row["prediction"]))


ID: bgv600_nordkorea_unanswerable Language: ar
Pair: bgv600_nordkorea
Question: أين تقع الدنمارك؟
Context: North Korea, officially the Democratic People's Republic of Korea (DPRK), is a country in East Asia.
Gold answerable: False
Gold answer: ''
Predicted answer: 'East Asia'

ID: bgv600_FrankieGaye_unanswerable Language: ar
Pair: bgv600_FrankieGaye
Question: متى توقف فرانكي عن العمل مع شقيقه؟
Context: Starting in the mid-1970s, Frankie began working with his brother, joining him on the road during Gaye's concert tours and sometimes, to test audience reactions, Marvin would place Frankie onstage first before he arrived.
Gold answerable: False
Gold answer: ''
Predicted answer: 'mid-1970s'

ID: bgv600_Flower_answerable Language: te
Pair: bgv600_Flower
Question: ఒలేరియా స్టెల్లులాటా అంటే ఏమిటి?
Context: Olearia stellulata is a species of flowering plant in the family Asteraceae and is endemic to south-eastern Australia. It is a shrub with lance-shaped or narrowly elliptic leaves, and whi